# Markov Models

## Map of Models

The decision-making models in this course grow out of a small family of probabilistic models.

![map-of-models](images/map-of-models.png)

## Today

Passive models known as **stochastic processes**:
- No actions, control, or decisions (we only watch a system evolve).
- Temporal probabilistic dynamics (except Bayes nets).

![today](images/today-models.png)

## The Markov Assumption

<img src="images/markov.jpg" alt="Andrey Markov (1856-1922)" width="140" align="right"/>

The Markov assumption states that once the model's designated local state (or context) is known, the variable of interest is independent of the rest of the past, or of nonlocal variables.

$$\text{Future} \perp \text{Past} \mid \text{Present}$$

<p style="text-align: right; font-size: 0.8em">Andrey Markov (1856&ndash;1922)</p>

For a sequence of random variables $X_0, X_1, X_2, \dots$ this reads

$$P(X_{t+1} \mid X_0, X_1, \dots, X_t) = P(X_{t+1} \mid X_t).$$

The present state summarizes everything about the past that matters for the future.

## Running Example: Office Delivery Robot

<img src="images/delivery-robot.png" alt="delivery robot" width="300" align="right"/>

- A robot delivers letters and packages to employees in an office building.
- It moves between rooms to find the recipients.
- We will model this robot with each of today's probabilistic models, to compare their expressiveness and properties.

_Question_: We start by describing the robot only by its location. Is the location alone a Markov state? What could make the next location depend on more than the current one?

## Markov Chains

A Markov chain is a tuple $\langle \mathcal{X}, P, \mu_0 \rangle$:
- $\mathcal{X}$ is a set of states (or events). In our case, $\mathcal{X}$ is finite with size $n$. The state at time $t$ is a random variable $X_t \in \mathcal{X}$.
- $P$ is an $n \times n$ matrix of transition probabilities, $P_{ij} = P(X_{t+1} = j \mid X_t = i)$. Row $i$ is a distribution over the next state, so every row sums to $1$. **Transitions are Markov.**
- $\mu_0$ is the initial distribution over $\mathcal{X}$. The first state is sampled $X_0 \sim \mu_0$.

We write distributions as row vectors, where $\mu_0[i] = P(X_0 = i)$ and, similarly, $\mu_t[i] = P(X_t = i)$.

### The robot as a Markov chain

Suppose our robot collects mail from the mailroom and delivers it to one of two offices. We want to know how the robot spends its time.

- $\mathcal{X} = \{\text{at-mailroom}, \text{at-office1}, \text{at-office2}\}$
- $P = \begin{pmatrix} 0.2 & 0.4 & 0.4 \\ 0.4 & 0.1 & 0.5 \\ 0.7 & 0.2 & 0.1 \end{pmatrix}$, where row $i$ says where the robot goes next from room $i$.
- $\mu_0 = (1, 0, 0)$, i.e., the robot starts in the mailroom.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

rooms = ["at-mailroom", "at-office1", "at-office2"]

# row i is the distribution over the next room, given that the robot is in room i
P = np.array([
    [0.2, 0.4, 0.4],   # from mailroom
    [0.4, 0.1, 0.5],   # from office1
    [0.7, 0.2, 0.1],   # from office2
])
mu0 = np.array([1.0, 0.0, 0.0])   # start in the mailroom

assert np.allclose(P.sum(axis=1), 1)   # every row is a distribution

### Sampling the chain

A Markov chain is a simulator. To generate the next state, sample from the row of the current state.

In [ ]:
def sample_chain(P, mu0, n_steps, rng):
    x = rng.choice(len(mu0), p=mu0)          # X_0 ~ mu_0
    states = [x]
    for t in range(n_steps):
        x = rng.choice(len(P), p=P[x])       # X_{t+1} ~ P(. | X_t = x): row x of P
        states.append(x)
    return np.array(states)


rng = np.random.default_rng(0)
print("   " + "\n-> ".join(rooms[x] for x in sample_chain(P, mu0, n_steps=10, rng=rng)))

_Question_: How can we compute the probability that the robot is in office2 at $t = 2$ exactly, without sampling?

## Distribution Propagation

By the law of total probability,

$$P(X_{t+1} = i) = \sum_{j=1}^{n} P(X_t = j)\, P(X_{t+1} = i \mid X_t = j) = \sum_{j=1}^{n} \mu_t[j]\, P_{ji}.$$

This is the vector-matrix product $\mu_{t+1} = \mu_t P$, and therefore $\mu_t = \mu_0 P^t$.

In [ ]:
def propagate(P, mu0, n_steps):
    mus = [mu0]
    for t in range(n_steps):
        mus.append(mus[-1] @ P)   # mu_{t+1} = mu_t P
    return np.array(mus)


mus = propagate(P, mu0, n_steps=15)
print("mu_1 =", mus[1])
print("mu_2 =", mus[2])

# sanity check against the simulator: where are 10,000 sampled robots at t = 2?
at_t2 = [sample_chain(P, mu0, n_steps=2, rng=rng)[-1] for _ in range(10_000)]
print("fraction of sampled robots in each room at t = 2:", np.bincount(at_t2, minlength=3) / len(at_t2))

### Where is the robot after $t$ steps?

In [ ]:
import matplotlib.pyplot as plt

from utils import plot_distribution_evolution

plt.figure(figsize=(8, 4))
plot_distribution_evolution(mus, rooms)
plt.show()

After a handful of steps the distribution stops changing. The robot is in the mailroom with probability of about $0.42$, no matter how many more steps we take.

_Question_: Does this limit depend on where the robot starts?

## Stationary Distributions

A state distribution $\mu$ is called **stationary** if $\mu P = \mu$, i.e., propagation leaves the distribution unchanged.

**Theorem**: Every finite Markov chain has at least one stationary distribution.

**Theorem**: If a finite Markov chain is *irreducible* (every state can reach every other state) and *aperiodic* (returns to a state are not locked to multiples of some period larger than $1$), then it has a unique stationary distribution $\mu$, and $\mu_t \xrightarrow[t \to \infty]{} \mu$ for **any** initial distribution $\mu_0$ (Freedman, 2017).

### Computing $\mu$

$\mu P = \mu$ says that $\mu$ is a left eigenvector of $P$ with eigenvalue $1$, i.e., $(P^\top - I)\,\mu^\top = 0$. Adding the constraint $\sum_i \mu[i] = 1$ gives a linear system with a unique solution (for chains that satisfy the theorem).

In [ ]:
def stationary_distribution(P):
    n = len(P)
    # stack the equations (P^T - I) mu = 0 and sum(mu) = 1, then solve them together
    A = np.vstack([P.T - np.eye(n), np.ones(n)])
    b = np.append(np.zeros(n), 1.0)
    return np.linalg.lstsq(A, b, rcond=None)[0]


mu = stationary_distribution(P)
print("stationary distribution:", mu)
print("mu P - mu =", mu @ P - mu)

The robot chain converges to $\mu \approx (0.4152, 0.2573, 0.3275)$.

### Does the start matter?

The robot chain is irreducible, and it is aperiodic because the robot can stay in the same room for a step. The theorem promises the same limit from every start.

In [ ]:
from utils import plot_start_comparison

# P(X_t = mailroom) when the robot starts in each of the three rooms
curves = {room: propagate(P, np.eye(3)[i], n_steps=12)[:, 0] for i, room in enumerate(rooms)}

plt.figure(figsize=(8, 3.8))
plot_start_comparison(curves, "mailroom")
plt.axhline(mu[0], color="k", linestyle="--", linewidth=1, label=r"stationary $\mu$[mailroom]")
plt.legend(fontsize=8)
plt.show()

### How the robot spends its time

$\mu$ also answers the question we started with. For an irreducible chain, the fraction of time that a *single* long trajectory spends in state $i$ converges to $\mu[i]$.

In [ ]:
from utils import plot_time_fractions

states = sample_chain(P, mu0, n_steps=10_000, rng=rng)

plt.figure(figsize=(8, 4))
plot_time_fractions(states, mu, rooms)
plt.show()
print("fraction of time in each room:", np.bincount(states, minlength=3) / len(states))

**The robot spends over 40% of its time waiting for mail in the mailroom!**

### When convergence fails

_Question_: Can you build a three-room chain that has a stationary distribution, yet $\mu_t$ never converges to it?

Two operating rules that each break one condition of the theorem:
- A **strict schedule** (periodic), where the robot always goes mailroom $\to$ office1 $\to$ office2 $\to$ mailroom.
- **Locked doors** (reducible), where the corridor between the mailroom and the offices is closed. A robot in the mailroom stays there, and a robot in the offices moves only between them.

In [ ]:
P_periodic = np.array([
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0],
])
P_reducible = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.3, 0.7],
    [0.0, 0.6, 0.4],
])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, chain, title in [(axes[0], P_periodic, "strict schedule (periodic)"),
                         (axes[1], P_reducible, "locked doors (reducible)")]:
    curves = {room: propagate(chain, np.eye(3)[i], n_steps=12)[:, 0] for i, room in enumerate(rooms)}
    plot_start_comparison(curves, "mailroom", ax=ax, title=title)
plt.tight_layout()
plt.show()

print("strict schedule, stationary distribution:", stationary_distribution(P_periodic))
for candidate in [np.array([1.0, 0.0, 0.0]), np.array([0.0, 6 / 13, 7 / 13])]:
    print("locked doors,", candidate, "stationary?", np.allclose(candidate @ P_reducible, candidate))

The strict schedule has the stationary distribution $(\frac13, \frac13, \frac13)$, but $\mu_t$ cycles forever and never reaches it. With locked doors there are *two* stationary distributions, and the limit is decided by where the robot started.

Note that aperiodic does not mean "no cycles". The robot chain is full of cycles (mailroom $\to$ office1 $\to$ mailroom), but none of them sets a fixed rhythm. In an irreducible chain, a single state that can repeat itself ($P_{ii} > 0$) is enough to rule out any period.

## The Supervisor's Decision

The supervisor is not happy that the robot spends over 40% of its time in the mailroom, and considers two new operating rules:
- **Rule A** (no loitering) makes the robot leave the mailroom right after collecting mail. The mailroom row becomes $(0, 0.5, 0.5)$.
- **Rule B** (batch deliveries) makes the robot continue from office2 to office1 more often, instead of returning to the mailroom. The office2 row becomes $(0.35, 0.55, 0.1)$.

_Question_: Which rule cuts the robot's long-run time in the mailroom more?

In [ ]:
from utils import plot_distribution_bars

P_rule_a = P.copy()
P_rule_a[0] = [0.0, 0.5, 0.5]        # leave the mailroom immediately
P_rule_b = P.copy()
P_rule_b[2] = [0.35, 0.55, 0.1]      # from office2, return to the mailroom less often

plt.figure(figsize=(8, 4))
plot_distribution_bars({"current": mu,
                        "rule A (no loitering)": stationary_distribution(P_rule_a),
                        "rule B (batch deliveries)": stationary_distribution(P_rule_b)}, rooms)
plt.show()

# where does the mailroom's probability come from? flow into it from each room, per step
flow = mu * P[:, 0]
print("flow into the mailroom, by source room:", ", ".join(f"{r} {f:.3f}" for r, f in zip(rooms, flow)))

Rule A goes after the obvious target, the mailroom's own row, and only brings the mailroom down from 41.5% to 36.2%. Rule B never touches the mailroom row and brings it down to 31.9%. The long-run share of a state depends on how often the chain *enters* it, not only on how long it stays, and more than half of the flow into the mailroom arrives from office2.

Notice that the supervisor just made a **decision**, and each option selected a different transition matrix. A Markov chain has no place for choices, since it only describes what happens. Making the choice part of the model, with one transition matrix per action, gives the Markov decision process at the top of the map.

## Hidden Markov Models

In a hidden Markov model (HMM), the true state is unknown (hidden), but we can make observations that depend on it.

An HMM is a tuple $\langle \mathcal{X}, \mathcal{Z}, P, O, \mu_0 \rangle$:
- $\mathcal{X}$, $P$, and $\mu_0$ are as in Markov chains.
- $\mathcal{Z}$ is a set of possible observations. The observation at time $t$ is a random variable $Z_t \in \mathcal{Z}$.
- $O$ is the emission (observation) matrix, $O_{ik} = P(Z_t = k \mid X_t = i)$. **Emissions are Markov** (depend only on the current state).

### Inference tasks

What are some interesting questions that HMMs can answer? Given the observations $z_{1:t}$:
- **Filtering** estimates the current state, i.e., computes $P(X_t \mid z_{1:t})$.
- **Smoothing** estimates a past state, i.e., computes $P(X_k \mid z_{1:t})$ for some $k < t$.
- The **most likely sequence** is the sequence of states that best explains the observations, i.e., $\arg\max_{x_{0:t}} P(x_{0:t} \mid z_{1:t})$.

We implement filtering as this is the task an agent faces when it must make decisions when the true state is unknown. 

## Belief Filtering

Recursive filtering maintains a **belief** over the current state. Given the observations $z_{1:t}$, denote $\beta_t(x) = P(X_t = x \mid z_{1:t})$, and initialize $\beta_0 = \mu_0$.

**Predict** by pushing the belief through the dynamics, exactly as in Markov chain propagation,

$$\bar\beta_{t+1} = \beta_t P.$$

**Update** by reweighting each state according to how well it explains the new observation $z_{t+1}$ (Bayes' rule),

$$\beta_{t+1}(x) = \eta\, P(z_{t+1} \mid x)\, \bar\beta_{t+1}(x), \qquad \eta = \frac{1}{\sum_{x' \in \mathcal{X}} P(z_{t+1} \mid x')\, \bar\beta_{t+1}(x')}.$$

The normalizer $\eta$ ensures $\sum_{x} \beta_{t+1}(x) = 1$. Its inverse is the probability of the new observation given the past ones, $\sum_{x'} P(z_{t+1} \mid x')\, \bar\beta_{t+1}(x') = P(z_{t+1} \mid z_{1:t})$.

In [ ]:
def predict(belief, P):
    return belief @ P                           # beta_bar_{t+1} = beta_t P


def update(belief_bar, O, z):
    unnormalized = belief_bar * O[:, z]         # P(z | x) * beta_bar(x), for every state x
    return unnormalized / unnormalized.sum()    # multiply by the normalizer eta


def belief_filter(P, O, mu0, observations):
    beliefs = [mu0]                             # beta_0 = mu_0
    for z in observations:
        beliefs.append(update(predict(beliefs[-1], P), O, z))
    return np.array(beliefs)

## HMM Example

The supervisor wants to check whether the robot is slacking off. The supervisor does not know where the robot is, but can monitor what the robot's camera sees.

- $\mathcal{Z} = \{\text{seeMailBin}, \text{seeDesk}, \text{seeWaterCooler}\}$
- $O = \begin{pmatrix} 0.8 & 0.05 & 0.15 \\ 0.1 & 0.7 & 0.2 \\ 0.05 & 0.25 & 0.7 \end{pmatrix}$, with one row per room and one column per observation.
- We know the robot started in the mailroom, so $\beta_0 = \mu_0 = (1, 0, 0)$.

In [ ]:
observations = ["seeMailBin", "seeDesk", "seeWaterCooler"]

# row i is the distribution over what the robot sees, given that it is in room i
O = np.array([
    [0.8,  0.05, 0.15],   # in mailroom
    [0.1,  0.7,  0.2],    # in office1
    [0.05, 0.25, 0.7],    # in office2
])
assert np.allclose(O.sum(axis=1), 1)

### One step by hand

The robot reports $\text{seeDesk}$ at time $t = 1$.

- Predict $\bar\beta_1 = \beta_0 P = (0.2, 0.4, 0.4)$.
- Update with the observation, $P(z_1 \mid x) \cdot \bar\beta_1(x) = (0.2 \cdot 0.05,\ 0.4 \cdot 0.7,\ 0.4 \cdot 0.25) = (0.01, 0.28, 0.1)$.
- Normalize with $\eta = \frac{1}{0.01 + 0.28 + 0.1} = \frac{1}{0.39}$,

$$\beta_1 = \frac{(0.01, 0.28, 0.1)}{0.39} = (0.0256, 0.7179, 0.2564)$$

In [ ]:
z1 = observations.index("seeDesk")
belief_bar = predict(mu0, P)
print("predict:", belief_bar)
print("update: ", update(belief_bar, O, z1))

The robot is likely delivering mail, and not slacking off in the mailroom.

_Question_: At $t = 2$ the robot reports $\text{seeMailBin}$. Where does the belief move, and how sure should the supervisor be?

In [ ]:
beliefs = belief_filter(P, O, mu0, [z1, observations.index("seeMailBin")])
print("beta_2 =", beliefs[2])

A single $\text{seeMailBin}$ puts over 90% of the belief on the mailroom. The mailroom emits it with probability $0.8$, while the offices almost never do.

## Tracking the Robot

Now let the robot run for a while, and let the supervisor track it. Sampling an HMM is sampling a Markov chain, plus one observation per step from the row of $O$ that belongs to the current state.

In [ ]:
from utils import plot_belief

def sample_hmm(P, O, mu0, n_steps, rng):
    x = rng.choice(len(mu0), p=mu0)
    states, obs = [x], []
    for t in range(n_steps):
        x = rng.choice(len(P), p=P[x])               # hidden:  X_{t+1} ~ P(. | X_t)
        states.append(x)
        obs.append(rng.choice(O.shape[1], p=O[x]))   # visible: Z_{t+1} ~ O(. | X_{t+1})
    return np.array(states), np.array(obs)


rng = np.random.default_rng(1)
true_rooms, zs = sample_hmm(P, O, mu0, n_steps=60, rng=rng)
beliefs = belief_filter(P, O, mu0, zs)

plt.figure(figsize=(12, 2.8))
plot_belief(beliefs, true_rooms, rooms, title=r"the supervisor's belief $\beta_t$")
plt.show()
hits = np.mean(beliefs[1:].argmax(axis=1) == true_rooms[1:])
print(f"the most likely room under the belief is the true room in {hits:.0%} of the steps")

Most of the time the belief is confident and right. Almost every miss in this run (13 of 14) is a step where the camera saw something typical of *another* room, such as a desk from office2 or a water cooler from the mailroom, and the belief followed the observation. The prediction step was negligible in comparison.

### Does memory help?

The filter combines two sources of information:
- **Memory**, which the prediction step carries forward through the dynamics. Given where the robot was, where can it be now?
- **Evidence**, which the update step takes from the newest observation. Which room best explains what the camera sees?

A supervisor with no memory could skip the prediction, and just guess the room that best explains the latest observation, $\arg\max_x P(z_t \mid x)$. Is the filter's memory worth anything?

_Question_: When should the filter beat this memoryless guess by a wide margin, and when should the two be about the same?

In [ ]:
def compare_to_memoryless(P, O, mu0, n_steps=2000, n_seeds=10):
    filter_hits, memoryless_hits, changed = [], [], []
    for seed in range(n_seeds):
        states, zs = sample_hmm(P, O, mu0, n_steps, np.random.default_rng(seed))
        filter_guess = belief_filter(P, O, mu0, zs)[1:].argmax(axis=1)
        memoryless_guess = O[:, zs].argmax(axis=0)               # the best room for z_t alone
        filter_hits.append(np.mean(filter_guess == states[1:]))
        memoryless_hits.append(np.mean(memoryless_guess == states[1:]))
        changed.append(np.mean(filter_guess != memoryless_guess))   # memory overruled the evidence
    return np.mean(filter_hits), np.mean(memoryless_hits), np.mean(changed)


filter_acc, memoryless_acc, changed = compare_to_memoryless(P, O, mu0)
print(f"correct room:  filter {filter_acc:.3f}   memoryless {memoryless_acc:.3f}")
print(f"memory changed the guess in {changed:.1%} of the steps")

A tie. Memory changed the guess in only 2% of the steps. To see why, write the update in odds form. For any two rooms $x$ and $x'$,

$$\underbrace{\frac{\beta_t(x)}{\beta_t(x')}}_{\text{belief}} = \underbrace{\frac{\bar\beta_t(x)}{\bar\beta_t(x')}}_{\text{memory}} \times \underbrace{\frac{P(z_t \mid x)}{P(z_t \mid x')}}_{\text{evidence}}.$$

The memoryless guess follows the evidence alone. The filter overrules it only when memory favors another room by more than the evidence favors the guess. For example, $\text{seeWaterCooler}$ favors office2 over office1 by $0.7 / 0.2 = 3.5$ to $1$, so the filter answers office1 only if its prediction favors office1 by more than $3.5$ to $1$.

Our robot's prediction is rarely that confident. The robot moves at almost every step, and from every room it has at least two likely destinations, so even knowing exactly where it was says little about where it is now.

**Memory helps only when the past is a strong predictor of the present.**

### A sticky robot

Our model is coarse in time. One step is roughly one trip, so the robot changes rooms at almost every step. A camera, however, records many frames per trip. Between two consecutive frames, the robot is most likely still in the same room, and the past predicts the present well. Most tracking problems look like this, since sensors such as cameras and GPS sample the world much faster than it changes.

We model a faster camera with a **stickiness** parameter $s \in [0, 1)$. At each step, the robot stays in its room with probability $s$, and otherwise takes a step of the original chain,

$$P_s = sI + (1 - s)P.$$

On average, the robot follows the original chain once every $1 / (1 - s)$ frames (e.g., once every $10$ frames at $s = 0.9$), and stays put in between.

Stickiness makes the robot more predictable without changing where it spends its time. The stationary distribution is unchanged,

$$\mu P_s = s\,\mu + (1 - s)\,\mu P = \mu,$$

so the memoryless guess, which judges each frame on its own, has the same long-run accuracy for every $s$. Any gap that opens between the two supervisors is the value of memory alone.

### Memory overrules a misleading frame

The supervisor is sure that the robot was in office1, and the next frame shows a water cooler, which is typical of office2.

In [ ]:
was_in_office1 = np.array([0.0, 1.0, 0.0])
z = observations.index("seeWaterCooler")
for s in [0.0, 0.9]:
    P_s = s * np.eye(3) + (1 - s) * P
    prediction = predict(was_in_office1, P_s)
    print(f"s = {s}:  prediction {prediction}   belief {update(prediction, O, z)}")

At $s = 0$, the filter agrees with the camera, since the robot most likely did move to office2. At $s = 0.9$, the prediction favors office1 over office2 by $0.91 / 0.05 \approx 18$ to $1$, far more than the camera's $3.5$ to $1$, and the filter overrules the camera. A misleading frame is now five times more likely than a move ($0.91 \cdot 0.2$ against $0.05 \cdot 0.7$).

### How sticky must the robot be?

In [ ]:
stickiness = [0.0, 0.25, 0.5, 0.75, 0.9, 0.95]
results = np.array([compare_to_memoryless(s * np.eye(3) + (1 - s) * P, O, mu0) for s in stickiness])

plt.figure(figsize=(7, 4))
plt.plot(stickiness, results[:, 0], color="k", marker="o", linewidth=2, label="belief filter")
plt.plot(stickiness, results[:, 1], color="gray", marker="s", linewidth=2, linestyle="--",
         label="memoryless guess")
plt.xlabel("stickiness $s$")
plt.ylabel("fraction of steps with the correct room")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
for s, (f, m, c) in zip(stickiness, results):
    print(f"s = {s:.2f}:  filter {f:.3f}   memoryless {m:.3f}   memory changed the guess in {c:4.0%} of the steps")

While the robot moves at almost every step ($s \leq 0.5$), memory rarely changes the guess, and the two supervisors tie. Once the robot lingers ($s \geq 0.75$), the prediction becomes confident enough to overrule misleading frames, and the filter pulls away, reaching $0.88$ against $0.74$ at $s = 0.95$. The memoryless guess stays flat up to sampling noise, as the unchanged stationary distribution promised.

**When is memory needed?** When the past predicts the present better than a single observation does. Memory needs predictable dynamics to have something to say, and noisy observations to have something to correct. With a perfect camera, the filter has nothing to add, however sticky the robot is.

In [ ]:
from ipywidgets import FloatSlider, interact


# continuous_update=False: the long-run comparison takes a moment, so recompute only on release
@interact(s=FloatSlider(value=0.9, min=0.0, max=0.95, step=0.05, continuous_update=False))
def show_sticky_robot(s):
    P_s = s * np.eye(3) + (1 - s) * P
    states, zs = sample_hmm(P_s, O, mu0, n_steps=60, rng=np.random.default_rng(1))
    plt.figure(figsize=(12, 2.8))
    plot_belief(belief_filter(P_s, O, mu0, zs), states, rooms, title=f"stickiness $s = {s:.2f}$")
    plt.show()
    filter_acc, memoryless_acc, _ = compare_to_memoryless(P_s, O, mu0)   # the same runs as the sweep
    gap = round(filter_acc - memoryless_acc, 3) + 0.0                     # + 0.0 prints -0.000 as +0.000
    print(f"correct room in the long run:  filter {filter_acc:.3f}   memoryless {memoryless_acc:.3f}   "
          f"difference {gap:+.3f}")

## Bayesian Networks

What if we have more than one variable? How can we represent the relationships between them?

**Bayesian networks** are directed acyclic graphs (DAGs) where each node represents a random variable and each edge represents a direct conditional dependence between two variables. A Bayesian network represents a **single time slice** (it is non-temporal).

The graph provides a factorization of the joint distribution. For random variables $X_1, \dots, X_m$ arranged in a DAG,

$$P(X_1, \dots, X_m) = \prod_{i=1}^{m} P\big(X_i \mid \text{Parents}(X_i)\big).$$

Each variable needs only a table conditioned on its parents, called a **conditional probability table** (CPT).

_Question_: Why is this Markov?

### The robot as a Bayesian network

<div style="display: flex; align-items: center; gap: 4em">
<div>

There are now additional variables to consider:
- The robot location $L \in \{\text{mailroom}, \text{office1}, \text{office2}\}$
- The robot task mode $T \in \{\text{idle}, \text{delivery}\}$
- The robot's battery level $B \in \{0, 1, 2\}$
- The sensor status $S \in \{\text{working}, \text{notWorking}\}$

These variables have conditional relationships:
- The robot is only idle in the mailroom.
- A critical battery is likely to make the robot idle.
- The battery level affects the sensor status.
- Some locations have bad reception, which also affects the sensor.

</div>
<img src="images/bn-graph.png" alt="Bayesian network B -> T -> L, B -> S, L -> S" width="160" style="flex-shrink: 0"/>
</div>

This answers the question from the start. The location alone does not tell the whole story. Where the robot goes next depends on whether it is delivering, which depends on its battery.

### Conditional probability tables

<div style="display: flex; justify-content: center; align-items: flex-start; gap: 3em">
<div>

| $P(B)$ | $B = 0$ | $B = 1$ | $B = 2$ |
| --- | --- | --- | --- |
| | 0.33 | 0.33 | 0.34 |

</div>
<div>

| $P(T \mid B)$ | $B = 0$ | $B = 1$ | $B = 2$ |
| --- | --- | --- | --- |
| idle | 1 | 0.4 | 0.1 |
| delivery | 0 | 0.6 | 0.9 |

</div>
<div>

| $P(L \mid T)$ | idle | delivery |
| --- | --- | --- |
| mailroom | 1 | 0.05 |
| office1 | 0 | 0.55 |
| office2 | 0 | 0.4 |

</div>
</div>

| $P(S \mid B, L)$ | 0, mailroom | 0, office1 | 0, office2 | 1, mailroom | 1, office1 | 1, office2 | 2, mailroom | 2, office1 | 2, office2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| working | 0 | 0 | 0 | 0.5 | 0.75 | 0.6 | 0.6 | 0.99 | 0.8 |
| notWorking | 1 | 1 | 1 | 0.5 | 0.25 | 0.4 | 0.4 | 0.01 | 0.2 |

In code, we index every table by the parents first and the variable last, so that each row is a distribution (the transpose of the tables above).

In [ ]:
modes = ["idle", "delivery"]
sensor = ["working", "notWorking"]
idle, delivery = 0, 1

P_B = np.array([0.33, 0.33, 0.34])            # P(B)

P_T_given_B = np.array([                      # P(T | B), indexed [b, t]
    [1.0, 0.0],                               # b = 0
    [0.4, 0.6],                               # b = 1
    [0.1, 0.9],                               # b = 2
])

P_L_given_T = np.array([                      # P(L | T), indexed [t, l]
    [1.0,  0.0,  0.0],                        # idle: always in the mailroom
    [0.05, 0.55, 0.4],                        # delivery
])

P_S_given_BL = np.array([                     # P(S | B, L), indexed [b, l, s]
    [[0.0, 1.0], [0.0, 1.0],   [0.0, 1.0]],   # b = 0: the sensor never works
    [[0.5, 0.5], [0.75, 0.25], [0.6, 0.4]],   # b = 1
    [[0.6, 0.4], [0.99, 0.01], [0.8, 0.2]],   # b = 2
])

### The joint distribution

The factorization tells us how to compute any entry of the joint distribution,

$$P(b, t, l, s) = P(b)\, P(t \mid b)\, P(l \mid t)\, P(s \mid b, l).$$

In [ ]:
import itertools

def joint(b, t, l, s):
    return P_B[b] * P_T_given_B[b, t] * P_L_given_T[t, l] * P_S_given_BL[b, l, s]


# the full joint table: one entry per assignment (b, t, l, s)
J = np.zeros((3, 2, 3, 2))
for b, t, l, s in itertools.product(range(3), range(2), range(3), range(2)):
    J[b, t, l, s] = joint(b, t, l, s)

print("number of assignments:", J.size, "  total probability:", J.sum())

## Inference by Enumeration

Assume the battery level is $b = 1$, the task mode is $t = \text{delivery}$, and the sensor is in status $s = \text{notWorking}$. What is the probability that the robot is in the mailroom?

$$P(l \mid b, t, s) = \frac{P(l, b, t, s)}{P(b, t, s)} = \frac{P(l, b, t, s)}{\sum_{l'} P(l', b, t, s)}$$

Factorizing both using the Bayesian network gives

$$P(l \mid b, t, s) = \frac{P(b)\,P(t \mid b)\,P(l \mid t)\,P(s \mid l, b)}{\sum_{l'} P(b)\,P(t \mid b)\,P(l' \mid t)\,P(s \mid l', b)} = \frac{P(l \mid t)\,P(s \mid l, b)}{\sum_{l'} P(l' \mid t)\,P(s \mid l', b)}$$

The factors $P(b)\,P(t \mid b)$ do not depend on the location, so they cancel.

_Question_: Does learning that the sensor is not working make the mailroom more or less likely than the $0.05$ given by $P(L = \text{mailroom} \mid T = \text{delivery})$?

In [ ]:
b_obs, t_obs, s_obs = 1, delivery, sensor.index("notWorking")

numerators = J[b_obs, t_obs, :, s_obs]       # P(l, b, t, s) for every location l
posterior = numerators / numerators.sum()    # divide by P(b, t, s) = sum over l'

print("P(l, b, t, s):        ", ", ".join(f"{r} {p:.5f}" for r, p in zip(rooms, numerators)))
print(f"P(b, t, s):            {numerators.sum():.6f}")
print("P(l | b=1, t=delivery, s=notWorking):", ", ".join(f"{r} {p:.4f}" for r, p in zip(rooms, posterior)))

Exactly as derived, $P(\text{mailroom}, b, t, s) = 0.00495$ and $P(b, t, s) = 0.063855$, so $P(\text{mailroom} \mid b, t, s) \approx 0.0775$. This is the same ratio as $0.025 / 0.3225$, the two sums after cancelling $P(b)\,P(t \mid b) = 0.198$.

Learning that the sensor is broken raised the mailroom from $0.05$ to $0.0775$, and office2 from $0.4$ to $0.496$, the rooms where the sensor fails most often. It lowered office1, where the sensor fails least, from $0.55$ to $0.426$. Evidence about a child ($S$) flows back to its parent ($L$).

### Sampling a Bayesian network

A Bayesian network is also a simulator. Sample each variable after its parents, in the order of the DAG (*ancestral sampling*). Keeping only the samples that agree with the evidence estimates the same query (*rejection sampling*).

In [ ]:
def sample_robot(rng):
    b = rng.choice(3, p=P_B)                   # the root first,
    t = rng.choice(2, p=P_T_given_B[b])        # then each variable given its sampled parents
    l = rng.choice(3, p=P_L_given_T[t])
    s = rng.choice(2, p=P_S_given_BL[b, l])
    return b, t, l, s


rng = np.random.default_rng(0)
samples = np.array([sample_robot(rng) for _ in range(100_000)])

# keep only the samples that agree with the evidence b = 1, t = delivery, s = notWorking
keep = (samples[:, 0] == b_obs) & (samples[:, 1] == t_obs) & (samples[:, 3] == s_obs)
estimate = np.bincount(samples[keep, 2], minlength=3) / keep.sum()
print(f"kept {keep.sum()} of {len(samples)} samples")
print("rejection-sampling estimate:", ", ".join(f"{r} {p:.4f}" for r, p in zip(rooms, estimate)))

The estimate agrees with the exact answer up to sampling noise. But only about 6,400 of the 100,000 samples survived. The evidence has probability $P(b, t, s) \approx 0.064$, so about 94% of the work was thrown away. The rarer the evidence, the more expensive rejection sampling gets.

### Why is this Markov?

In a Markov chain, the future is independent of the past given the present. In a Bayesian network, the graph decides what "the present" is. **Each variable is independent of its non-descendants given its parents** (the *local Markov property*). The factorization is exactly this assumption, applied once per node.

For example, the only parent of $L$ is $T$, and $B$ is not a descendant of $L$. The battery level does affect the location, through the task mode, but once the task mode is known it adds no information.

In [ ]:
P_BTL = J.sum(axis=3)                         # P(b, t, l): marginalize out S

# without its parent, the battery level changes our belief about the location...
P_BL = P_BTL.sum(axis=1)
print("P(L | B = b), one row per b:\n", P_BL / P_BL.sum(axis=1, keepdims=True))

# ...but once the parent T = delivery is known, it no longer does (b = 0 never delivers)
P_BL_delivery = P_BTL[1:, delivery, :]
print("P(L | B = b, T = delivery), rows b = 1, 2:\n",
      P_BL_delivery / P_BL_delivery.sum(axis=1, keepdims=True))

## Dynamic Bayesian Networks

A dynamic Bayesian network (DBN) extends a Bayesian network over time, as Markov chains and HMMs do for a single variable.
- A Bayesian network only considers one time slice.
- DBN = Bayesian network + temporal edges.

Transitions are defined by a **2 time-slice Bayesian network** (2-TBN) over the variables at time $t$ and the variables at time $t+1$, with edges into slice $t+1$ only. Each variable at $t+1$ has a CPT conditioned on its parents, from either slice.

<div style="display: flex; align-items: center; gap: 4em">
<div>

The HMM is a special case of a DBN:
- a single state variable $X_t$,
- a temporal edge, the transition matrix $P$,
- an observation edge, the emission matrix $O$.

</div>
<img src="images/hmm-as-dbn.png" alt="HMM as a DBN" width="220" style="flex-shrink: 0"/>
</div>

### The robot as a DBN

<div style="display: flex; align-items: center; gap: 3em">
<div style="flex-grow: 1; flex-basis: 0">

Each variable in slice $t+1$ has a CPT given its parents:
- **Battery**, $P(B_{t+1} \mid B_t, T_t)$. An idle robot charges (one level up w.p. $0.5$), and a delivering robot drains (one level down w.p. $0.2$).
- **Task mode**, $P(T_{t+1} \mid B_{t+1})$, as in the Bayesian network.
- **Location**, $P(L_{t+1} \mid L_t, T_t, T_{t+1})$. An idle robot sits at its charging dock in the mailroom, a delivering robot moves like the Markov chain from before, and a robot that just finished charging heads straight to an office.
- **Sensor**, $P(S_{t+1} \mid B_{t+1}, L_{t+1})$, as in the Bayesian network.
- **Observation**, $P(Z_{t+1} \mid L_{t+1}, S_{t+1})$. A working sensor sees as in the HMM ($O$), and a broken sensor reports a fourth observation, $\text{noSignal}$.

</div>
<img src="images/dbn-2tbn.png" alt="2-TBN of the delivery robot" width="340" style="flex-shrink: 0"/>
</div>

In [ ]:
P_B_next = np.zeros((3, 2, 3))                 # P(B' | B, T), indexed [b, t, b']
for b in range(3):
    P_B_next[b, idle, min(b + 1, 2)] += 0.5    # idle: charge one level...
    P_B_next[b, idle, b] += 0.5                # ...or not yet
    P_B_next[b, delivery, max(b - 1, 0)] += 0.2   # delivery: drain one level...
    P_B_next[b, delivery, b] += 0.8               # ...or not yet

P_L_next = np.zeros((3, 2, 2, 3))              # P(L' | L, T, T'), indexed [l, t, t', l']
for l in range(3):
    for t in range(2):
        P_L_next[l, t, idle] = [1.0, 0.0, 0.0]                              # idle now: at the dock
        P_L_next[l, t, delivery] = P[l] if t == delivery else [0.0, 0.5, 0.5]   # just charged: leave

observations_dbn = observations + ["noSignal"]
P_Z_given_LS = np.zeros((3, 2, 4))             # P(Z | L, S), indexed [l, s, z]
P_Z_given_LS[:, 0, :3] = O                     # working sensor: the HMM's emission matrix
P_Z_given_LS[:, 1, 3] = 1.0                    # broken sensor: noSignal

### From DBN to HMM

Any DBN can be flattened into an HMM whose state is the joint assignment of all the state variables, $X_t = (B_t, T_t, L_t, S_t)$. Every entry of the flat transition matrix is a product of the 2-TBN's tables,

$$P(x' \mid x) = P(b' \mid b, t)\; P(t' \mid b')\; P(l' \mid l, t, t')\; P(s' \mid b', l')$$

In [ ]:
flat_states = list(itertools.product(range(3), range(2), range(3), range(2)))   # all (b, t, l, s)
n = len(flat_states)

P_flat = np.zeros((n, n))
for i, (b, t, l, s) in enumerate(flat_states):
    for j, (b2, t2, l2, s2) in enumerate(flat_states):
        P_flat[i, j] = (P_B_next[b, t, b2] * P_T_given_B[b2, t2]
                        * P_L_next[l, t, t2, l2] * P_S_given_BL[b2, l2, s2])

O_flat = np.array([P_Z_given_LS[l, s] for b, t, l, s in flat_states])
mu0_flat = J.reshape(-1)        # slice 0 is the Bayesian network itself

assert np.allclose(P_flat.sum(axis=1), 1) and np.allclose(O_flat.sum(axis=1), 1)
print(f"flat HMM: {n} states, a {P_flat.shape[0]} x {P_flat.shape[1]} transition matrix")

_Question_: How many numbers does each representation need to specify the transitions?

In [ ]:
transition_tables = {"P(B' | B, T)": P_B_next, "P(T' | B')": P_T_given_B,
                     "P(L' | L, T, T')": P_L_next, "P(S' | B', L')": P_S_given_BL}
for name, table in transition_tables.items():
    print(f"{name:18s} {table.size:5d} entries")
n_factored = sum(table.size for table in transition_tables.values())
print(f"{'DBN total':18s} {n_factored:5d} entries")
print(f"{'flat matrix':18s} {P_flat.size:5d} entries  ({P_flat.size / n_factored:.1f} times larger)")

The flat matrix needs $36^2 = 1296$ numbers, while the 2-TBN needs $78$, about 17 times fewer. And the gap is not a constant factor. The flat state space is the *product* of the variables' domains, and the transition matrix squares it, while each table only grows with the number of parents of its variable.

In [ ]:
from utils import plot_parameter_scaling

m = np.arange(1, 21)
flat_sizes = (2.0 ** m) ** 2        # 2^m joint states, squared
factored_sizes = m * 2 ** 3 * 2     # m tables, each with 2^3 parent assignments x 2 values

plt.figure(figsize=(7, 4))
plot_parameter_scaling(m, flat_sizes, factored_sizes)
plt.show()
print(f"m = 20 binary variables, 3 parents each: flat {flat_sizes[-1]:.1e} entries, DBN {factored_sizes[-1]}")

## Filtering in the DBN

The flattened DBN *is* an HMM, so the `sample_hmm` and `belief_filter` functions we wrote for the HMM run on it unchanged. The supervisor can now track variables that the robot never reports.

_Question_: The robot never reports its battery level. Can the supervisor still track it, and which observations would give it away?

In [ ]:
rng = np.random.default_rng(3)
true_flat, zs = sample_hmm(P_flat, O_flat, mu0_flat, n_steps=80, rng=rng)
beliefs = belief_filter(P_flat, O_flat, mu0_flat, zs).reshape(-1, 3, 2, 3, 2)   # beta_t(b, t, l, s)

true_battery = np.array([flat_states[x][0] for x in true_flat])
true_location = np.array([flat_states[x][2] for x in true_flat])

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
plot_belief(beliefs.sum(axis=(1, 2, 4)), true_location, rooms, ax=axes[0],
            title="belief over the location")
plot_belief(beliefs.sum(axis=(2, 3, 4)), true_battery, ["B = 0", "B = 1", "B = 2"], ax=axes[1],
            title="belief over the battery level (never observed)")
plt.tight_layout()
plt.show()
print("observations:", " ".join(observations_dbn[z] for z in zs[:20]), "...")

In [ ]:
def dbn_tracking(n_steps=2000, n_seeds=10):
    # baseline: always guess the most common value under the stationary distribution
    mu_flat = stationary_distribution(P_flat).reshape(3, 2, 3, 2)
    best_guess = {"location": mu_flat.sum(axis=(0, 1, 3)).argmax(),
                  "battery": mu_flat.sum(axis=(1, 2, 3)).argmax()}
    hits = {"location": [], "battery": []}
    baseline = {"location": [], "battery": []}
    for seed in range(n_seeds):
        xs, zs = sample_hmm(P_flat, O_flat, mu0_flat, n_steps, np.random.default_rng(seed))
        beliefs = belief_filter(P_flat, O_flat, mu0_flat, zs).reshape(-1, 3, 2, 3, 2)[1:]
        truth = {"location": np.array([flat_states[x][2] for x in xs[1:]]),
                 "battery": np.array([flat_states[x][0] for x in xs[1:]])}
        marginal = {"location": beliefs.sum(axis=(1, 2, 4)), "battery": beliefs.sum(axis=(2, 3, 4))}
        for var in hits:
            hits[var].append(np.mean(marginal[var].argmax(axis=1) == truth[var]))
            baseline[var].append(np.mean(truth[var] == best_guess[var]))
    for var in hits:
        print(f"{var:9s} correct: filter {np.mean(hits[var]):.3f}   "
              f"best constant guess {np.mean(baseline[var]):.3f}")


dbn_tracking()

The location is tracked well. The filter names the right room in 75% of the steps, against 54% for always guessing the mailroom. The battery is tracked only loosely, and its belief is rarely confident, but it still beats the best constant guess (58% against 46%). The clues are indirect. A $\text{noSignal}$ is more likely at a low charge, and a run of $\text{seeMailBin}$ suggests that the robot is idle at its dock, which happens mostly at a low charge. A hidden variable can only be tracked as well as its footprint on the observations allows.

The factored model is compact to *write down*, but our filter still tracks a belief over all $36$ joint states, at a cost of $36^2$ operations per step. Exact filtering generally needs the full joint belief, since the variables become correlated through the observations. Filters that keep the belief factored are approximations, and a topic of their own (e.g., Boyen & Koller, 1998).

## Conclusion

We learned about three stochastic Markov processes:
- In **Markov chains**, distributions propagate as $\mu_{t+1} = \mu_t P$ and, for irreducible and aperiodic chains, converge to a unique stationary distribution $\mu$, which is also the long-run share of time in each state.
- In **hidden Markov models**, the state is hidden, and a belief filter tracks it by alternating *predict* (the dynamics) and *update* (Bayes' rule). The dynamics model is the filter's memory, and it pays off when the past predicts the present better than a single observation does.
- **Dynamic Bayesian networks** factor the state into variables with a Bayesian network, which makes the model exponentially smaller to specify. Any DBN can still be flattened into an HMM.

But these models **do not allow us to take action** and control the outcome of the system. We only watched the robot. The one time its behavior changed, the supervisor did it from outside the model by choosing a different transition matrix. From now on, we focus on *decision processes*:
- A Markov decision process (MDP) adds control to a Markov chain.
- A partially observable MDP (POMDP) adds control to an HMM.
- A factored (PO)MDP adds control to a DBN.

![map-of-models](images/map-of-models.png)